from google.colab import drive
drive.mount('/content/drive')

In [1]:
import pandas as pd
import os
#os.chdir('/content/drive/MyDrive/name/IMP-OIC-Windowing')
from utils.extractframes import FrameExtractor
import graphene
directory_path = 'TVQAP/video_frames/frames_hq/bbt_frames'
from gpt_ask import run_gpt
main_ds = pd.read_json('TVQAP/tvqa_with_subtitle.json')

main = main_ds
main_ds

,a1,qid,answer_idx,ts,q,a0,a3,a2,bbox,vid_name,a4,subtitle
0,Raj is weird .,133290,3,"[0.0, 5.4]",Why did Raj tell himself to turn his pelvis wh...,Raj was trying to get away from Penny .,Raj had become excited and did not want Penny ...,Raj likes to give himself odd instructions .,"{'1': [{'img_id': 1, 'top': 14, 'height': 346,...",s01e02_seg02_clip_12,Raj did not like hugging Penny .,UNKNAME : Uh oh . Turn your pelvis .
1,Howard told Leonard that he would never beat h...,136932,2,"[12.3, 20.11]",What did Howard tell Leonard after he finished...,Howard told Leonard to go pick up his lunch .,Howard told Leonard to compare their scores .,Howard told Leonard to grab a napking because ...,"{'39': [{'img_id': 39, 'top': 43, 'height': 24...",s01e02_seg02_clip_12,Howard told Leonard to copy his dance moves .,UNKNAME : Uh oh . Turn your pelvis .
2,Leonard told Howard that Howard is n't very go...,133451,3,"[17.71, 25.51]",What did Leonard tell Howard after Howard said...,Leonard told Howard that he really hates that ...,"Leonard told Howard that it was fine , he wins .",Leonard told Howard that Sheldon will beat his...,"{'68': [{'img_id': 68, 'top': 66, 'height': 29...",s01e02_seg02_clip_12,Leonard told Howard that he will beat him .,"UNKNAME : Grab a napkin , homey , you just go..."
3,Raj said that Penny was very happy .,134785,4,"[46.83, 55.84]",Why did Raj say that Penny was upset after Leo...,Raj said that Penny did n't mention Leonard .,Raj said that Penny was only mad at Howard .,Raj said that Penny was very mad at Leonard .,"{'155': [{'img_id': 155, 'top': 23, 'height': ...",s01e02_seg02_clip_12,Raj said that Penny was upset because Penny 's...,"UNKNAME : Grab a napkin , homey , you just go..."
4,Lesley says there was no passion .,131529,0,"[7.67, 16.23]",What does Lesley say there was none of when Le...,Lesley says there was no arousal .,Lesley says the kiss lacked a certain fire .,There was no kiss .,"{'26': [{'img_id': 26, 'top': 20, 'height': 34...",s01e03_seg02_clip_05,Lesley says there was no excitement in the kiss .,Lesley : . no extraneous spittle .
...,...,...,...,...,...,...,...,...,...,...,...,...
3012,Handed Leonard a mug .,124343,0,"[14.18, 24.14]",What did Sheldon do after telling Leonard he s...,Placed his mug into the microwave .,Opened a bag of cookies .,Took off his robe .,"{'39': [{'img_id': 39, 'top': 20, 'height': 34...",s01e10_seg02_clip_01,Opened a drawer .,Sheldon : . pressing on the cognitive process...
3013,She walks out of the apartment,131650,2,"[31.12, 44.55]",What does Beverley do after Leonard says he wa...,She slams her fist on the table,She starts crying,She hugs him,"{'114': [{'img_id': 114, 'top': 21, 'height': ...",s03e11_seg02_clip_15,She sits down next to him,
3014,Starts crying,132157,3,"[49.13, 56.76]",What does Leonard do after he hugs his mother ?,Starts yelling,Goes to bed,Walks out of the apartment,"{'150': [{'img_id': 150, 'top': 87, 'height': ...",s03e11_seg02_clip_15,Slams the door,
3015,Leonard takes his cell phone from his pocket .,122410,0,"[5.58, 15.51]",What does Leonard pull from his pocket after g...,Leonard pulls a handful of cash from his pocket .,Leonard takes box of condoms from his pocket .,Leonard takes a box of mints from his pocket .,"{'10': [], '22': [{'img_id': 22, 'top': 48, 'h...",s03e12_seg01_clip_00,Leonard takes his keys from his pocket .,


In [2]:
q_ids = main['qid'].unique()
for qid in q_ids:
  #if 'Seq' in q:
    que = main.query("qid=={}".format(qid))
    video_id = que['vid_name'].values[0]
    question = que['q'].values[0]
    answer_id = main.query("qid=={}".format(qid))["answer_idx"].values[0]
    subtitle = que['subtitle'].values[0]
    choice_string = ''

    choice_string = "0: {}, 1: {}, 2: {}, 3: {}, 4:{}".format(que['a0'].values[0], que['a1'].values[0],que['a2'].values[0],que['a3'].values[0],que['a4'].values[0])
    
    formatted_question = question+ 'Guess the most likely answer among these options: '+choice_string+' Respond only with a single number between 0 and 4. Do not produce any other output.'
    response = run_gpt('subtitle: '+subtitle, formatted_question)
    main.loc[main['qid'] == qid, 'OIC_answer'] = str(response)
    main.loc[main['qid'] == qid, 'OIC_question'] = formatted_question
    OIC_answer = response
    print(choice_string)
    print(OIC_answer)
   
    if len(OIC_answer)>1:
      main.loc[main['qid'] == qid, 'Match'] = OIC_answer
    else:
      if int(OIC_answer) == int(answer_id):
          main.loc[main['qid'] == qid, 'Match'] = 'Correct'
          print('correct')
      else:
          print('wrong')
          main.loc[main['qid'] == qid, 'Match'] = 'Wrong'
    print('-'*100)
    print('OIC question: {}'.format(formatted_question))
    print('OIC answer: {}'.format(OIC_answer))
    main.to_csv('TVQAP/baseline_gpt4_w_st.csv')
          
main.head()
main.name = 'all data'

0: Raj was trying to get away from Penny ., 1: Raj is weird ., 2: Raj likes to give himself odd instructions ., 3: Raj had become excited and did not want Penny to know ., 4:Raj did not like hugging Penny .
3
correct
----------------------------------------------------------------------------------------------------
OIC question: Why did Raj tell himself to turn his pelvis when Penny was giving him a hug ?Guess the most likely answer among these options: 0: Raj was trying to get away from Penny ., 1: Raj is weird ., 2: Raj likes to give himself odd instructions ., 3: Raj had become excited and did not want Penny to know ., 4:Raj did not like hugging Penny . Respond only with a single number between 0 and 4. Do not produce any other output.
OIC answer: 3
0: Howard told Leonard to go pick up his lunch ., 1: Howard told Leonard that he would never beat his score ., 2: Howard told Leonard to grab a napking because he just got served ., 3: Howard told Leonard to compare their scores ., 4:Ho

KeyboardInterrupt: 